<a href="https://colab.research.google.com/github/Nishant-ZFYII/AI_ML_Quants_VIP/blob/main/phase0_classification_upgrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase-0 Classifier Upgrade

Goal: push the 4 classification models **above 50% accuracy** using only the **original data** (Investing.com wheat OHLCV + FRED-MD). No external alt data.

Fixes applied in this notebook:
1. **OHLCV unlocked** — prior notebooks read only `Price` (close) and discarded Open/High/Low/Volume. This notebook reads all 5 columns.
2. **Technical indicators** — RSI(14), MACD(12,26,9), Bollinger %B(20), ATR(14), momentum(5/10/20), volume z-score.
3. **Target redefined** — from next-day direction (noisy, ~50% ceiling) to **5-day forward direction** (sign of 5-day return).
4. **Wavelet-denoised price** — Discrete Wavelet Transform smoothing before feature computation (Lopez Gil 2024).
5. **FRED-MD compressed** — monthly delta only, no 30-day duplicate rows.
6. **BiGRU pooling bug fixed** — `GlobalAveragePooling1D` → last hidden state + attention pooling.

Run order: top to bottom. Reports before/after accuracy for each of the 4 models.

In [1]:
!pip install -q PyWavelets

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import pywt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import warnings; warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)
print('Device:', DEVICE)

Device: cuda


In [4]:
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Quants ')
WHEAT_18 = BASE / 'Investing.com' / 'US Wheat Futures Historical Data_2018.csv'
WHEAT_25 = BASE / 'Investing.com' / 'US Wheat Futures Historical Data_2025.csv'
FRED = BASE / 'FredMD_Dataset' / '2025-10-MD.csv'

Mounted at /content/drive


## 1. Load full OHLCV (not just Close)

In [5]:
def parse_volume(v):
    if pd.isna(v): return np.nan
    v = str(v).replace(',', '').strip()
    if v in ('', '-'): return np.nan
    mult = 1.0
    if v.endswith('K'): mult, v = 1e3, v[:-1]
    elif v.endswith('M'): mult, v = 1e6, v[:-1]
    try: return float(v) * mult
    except: return np.nan

def load_ohlcv(p):
    df = pd.read_csv(p)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    for c in ['Price','Open','High','Low']:
        df[c] = df[c].astype(str).str.replace(',', '', regex=False).astype(float)
    df['Volume'] = df['Vol.'].apply(parse_volume)
    return df[['Price','Open','High','Low','Volume']].rename(columns={'Price':'Close'})

d1 = load_ohlcv(WHEAT_18); d2 = load_ohlcv(WHEAT_25)
ohlcv = pd.concat([d1, d2]).pipe(lambda d: d[~d.index.duplicated(keep='last')]).sort_index()
ohlcv['Volume'] = ohlcv['Volume'].ffill().bfill().fillna(ohlcv['Volume'].median())
print('OHLCV shape:', ohlcv.shape); print(ohlcv.tail(3))

OHLCV shape: (6864, 5)
            Close    Open    High     Low   Volume
Date                                              
2025-12-10  529.5  534.00  535.00  525.25  68630.0
2025-12-11  533.5  529.75  534.75  529.25  62420.0
2025-12-12  530.5  534.50  536.00  529.10  24570.0


## 2. Wavelet-denoise the close price

In [6]:
def wavelet_denoise(series, wavelet='db4', level=3):
    x = series.values.astype(float)
    coeffs = pywt.wavedec(x, wavelet, level=level)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    uthresh = sigma * np.sqrt(2 * np.log(len(x)))
    coeffs[1:] = [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs[1:]]
    out = pywt.waverec(coeffs, wavelet)[:len(x)]
    return pd.Series(out, index=series.index)

ohlcv['Close_dn'] = wavelet_denoise(ohlcv['Close'])
print('Denoise residual std:', float((ohlcv['Close'] - ohlcv['Close_dn']).std()))

Denoise residual std: 8.318318181546406


## 3. Technical indicators (from OHLCV + denoised close)

In [7]:
def rsi(s, n=14):
    d = s.diff(); up = d.clip(lower=0); dn = -d.clip(upper=0)
    ru = up.ewm(alpha=1/n, adjust=False).mean()
    rd = dn.ewm(alpha=1/n, adjust=False).mean()
    rs = ru / (rd + 1e-9)
    return 100 - 100/(1+rs)

def macd(s, f=12, sl=26, sig=9):
    ef = s.ewm(span=f, adjust=False).mean(); es = s.ewm(span=sl, adjust=False).mean()
    m = ef - es; sg = m.ewm(span=sig, adjust=False).mean()
    return m, sg, m - sg

def bb_percent(s, n=20, k=2):
    ma = s.rolling(n).mean(); sd = s.rolling(n).std()
    up = ma + k*sd; lo = ma - k*sd
    return (s - lo) / (up - lo + 1e-9)

def atr(df, n=14):
    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift()).abs(),
        (df['Low']  - df['Close'].shift()).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/n, adjust=False).mean()

feat = pd.DataFrame(index=ohlcv.index)
c = ohlcv['Close_dn']
feat['ret_1']  = c.pct_change(1)
feat['mom_5']  = c.pct_change(5)
feat['mom_10'] = c.pct_change(10)
feat['mom_20'] = c.pct_change(20)
feat['rsi_14'] = rsi(c, 14)
m, sg, h = macd(c); feat['macd'] = m; feat['macd_sig'] = sg; feat['macd_hist'] = h
feat['bb_pct'] = bb_percent(c, 20)
feat['atr_14'] = atr(ohlcv, 14)
feat['atr_pct'] = feat['atr_14'] / ohlcv['Close']
feat['hl_range'] = (ohlcv['High'] - ohlcv['Low']) / ohlcv['Close']
feat['oc_ret'] = (ohlcv['Close'] - ohlcv['Open']) / ohlcv['Open']
vol_z = (ohlcv['Volume'] - ohlcv['Volume'].rolling(60).mean()) / (ohlcv['Volume'].rolling(60).std() + 1e-9)
feat['vol_z'] = vol_z
feat['vol_chg'] = ohlcv['Volume'].pct_change().clip(-5, 5)
feat['close_dn_gap'] = (ohlcv['Close'] - ohlcv['Close_dn']) / ohlcv['Close']
print('Technical features:', feat.shape, list(feat.columns))

Technical features: (6864, 16) ['ret_1', 'mom_5', 'mom_10', 'mom_20', 'rsi_14', 'macd', 'macd_sig', 'macd_hist', 'bb_pct', 'atr_14', 'atr_pct', 'hl_range', 'oc_ret', 'vol_z', 'vol_chg', 'close_dn_gap']


## 4. FRED-MD — monthly, transformed, NO forward-fill duplication

Old notebooks forward-filled monthly FRED-MD to daily, giving 30 duplicate rows in each lookback window. We keep it monthly and merge as **one row per month** tagged to month-end, plus monthly-delta features only. Within-month rows carry NaN → later we drop the macro features entirely from the sequence model inputs that don't benefit, and keep them only as **context** fed to the classifier head.

In [8]:
tcodes_row = pd.read_csv(FRED, nrows=1)
fred_raw = pd.read_csv(FRED, skiprows=[1])
fred_raw['sasdate'] = pd.to_datetime(fred_raw['sasdate'], format='%m/%d/%Y')
fred_raw = fred_raw.set_index('sasdate').sort_index()

keep = ['RPI','W875RX1','CMRMTSPLx','IPFPNSS','FEDFUNDS','TB3MS','GS10','BAA','AAA',
        'T10YFFM','T5YFFM','T1YFFM','BAAFFM','S&P 500','EXCAUSx','EXUSUKx','CPIAUCSL',
        'PPICMM','OILPRICEx','UMCSENTx']
keep = [k for k in keep if k in fred_raw.columns]
fred = fred_raw[keep].copy()
fred = fred.pct_change().replace([np.inf,-np.inf], np.nan)
fred.columns = [f'fred_{c}_d' for c in fred.columns]
# publish with 1-month lag to avoid look-ahead
fred.index = fred.index + pd.DateOffset(months=1)
fred_daily = fred.reindex(pd.date_range(fred.index.min(), ohlcv.index.max(), freq='D')).ffill()
print('FRED features (monthly deltas, daily-aligned):', fred_daily.shape)

FRED features (monthly deltas, daily-aligned): (24422, 20)


## 5. Target: **5-day forward direction**

In [9]:
H = 5
fwd_ret = ohlcv['Close'].pct_change(H).shift(-H)
y_h = (fwd_ret > 0).astype(int)
y_1 = (ohlcv['Close'].pct_change(1).shift(-1) > 0).astype(int)  # for before-vs-after baseline
print(f'y_{H}d balance:', y_h.mean().round(3), '   y_1d balance:', y_1.mean().round(3))

y_5d balance: 0.486    y_1d balance: 0.478


## 6. Build matrix, chronological split (no leakage)

In [11]:
X_full = feat.join(fred_daily, how='left').ffill()
X_full = X_full.replace([np.inf, -np.inf], np.nan)
data = X_full.join(y_h.rename('y'), how='inner').join(y_1.rename('y1'), how='inner').dropna()
print('Aligned dataset:', data.shape)
FEATURE_COLS = [c for c in data.columns if c not in ('y','y1')]
print('#features:', len(FEATURE_COLS))

# 70/15/15 chronological
n = len(data); i1 = int(n*0.70); i2 = int(n*0.85)
train = data.iloc[:i1]; val = data.iloc[i1:i2]; test = data.iloc[i2:]
print(f'train {train.index.min().date()}..{train.index.max().date()}  n={len(train)}')
print(f'val   {val.index.min().date()}..{val.index.max().date()}  n={len(val)}')
print(f'test  {test.index.min().date()}..{test.index.max().date()}  n={len(test)}')

scaler = StandardScaler().fit(train[FEATURE_COLS])
def to_flat(split, ycol='y'):
    return scaler.transform(split[FEATURE_COLS]), split[ycol].values

LOOKBACK = 20
def to_seq(split, ycol='y'):
    Xs = scaler.transform(split[FEATURE_COLS])
    ys = split[ycol].values
    Xseq, yseq = [], []
    for i in range(LOOKBACK, len(Xs)):
        Xseq.append(Xs[i-LOOKBACK:i]); yseq.append(ys[i])
    return np.array(Xseq, dtype=np.float32), np.array(yseq, dtype=np.int64)

Xtr_f, ytr_f = to_flat(train); Xva_f, yva_f = to_flat(val); Xte_f, yte_f = to_flat(test)
Xtr_s, ytr_s = to_seq(train);  Xva_s, yva_s = to_seq(val);  Xte_s, yte_s = to_seq(test)
print('flat:', Xtr_f.shape, '  seq:', Xtr_s.shape)

Aligned dataset: (6801, 38)
#features: 36
train 1999-10-22..2017-12-29  n=4760
val   2018-01-02..2021-12-09  n=1020
test  2021-12-10..2025-12-12  n=1021
flat: (4760, 36)   seq: (4740, 20, 36)


## 7. Before/after baseline — 1-day target (old setup)
This is the "before" number we're trying to beat.

In [12]:
def fit_logreg(Xtr, ytr, Xte, yte, tag=''):
    clf = LogisticRegression(max_iter=2000, C=0.5).fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:,1]
    return {'tag':tag,'acc':accuracy_score(yte, p>0.5),'auc':roc_auc_score(yte, p),
            'f1':f1_score(yte, p>0.5)}

_, y1tr = to_flat(train,'y1'); _, y1te = to_flat(test,'y1')
before = fit_logreg(Xtr_f, y1tr, Xte_f, y1te, tag='LogReg / 1-day (BEFORE)')
after_target = fit_logreg(Xtr_f, ytr_f, Xte_f, yte_f, tag='LogReg / 5-day (AFTER target fix)')
print(before); print(after_target)

{'tag': 'LogReg / 1-day (BEFORE)', 'acc': 0.7071498530852106, 'auc': np.float64(0.7784794969253113), 'f1': 0.6739367502726281}
{'tag': 'LogReg / 5-day (AFTER target fix)', 'acc': 0.7433888344760039, 'auc': np.float64(0.8018073264989702), 'f1': 0.7069351230425056}


## 8. Model 1 — ARX-style classifier (fixed: uses full feature set)

In [13]:
class ARXClassifier:
    def __init__(self, C=0.5): self.clf = LogisticRegression(max_iter=3000, C=C)
    def fit(self, X, y): self.clf.fit(X, y); return self
    def predict_proba(self, X): return self.clf.predict_proba(X)[:,1]

m = ARXClassifier().fit(Xtr_f, ytr_f)
p_tr = m.predict_proba(Xtr_f); p_te = m.predict_proba(Xte_f)
res_arx = {'train_acc': accuracy_score(ytr_f, p_tr>0.5),
           'test_acc':  accuracy_score(yte_f, p_te>0.5),
           'test_auc':  roc_auc_score(yte_f, p_te),
           'test_f1':   f1_score(yte_f, p_te>0.5)}
print('ARX:', res_arx)

ARX: {'train_acc': 0.7418067226890757, 'test_acc': 0.7433888344760039, 'test_auc': np.float64(0.8018073264989702), 'test_f1': 0.7069351230425056}


## 9. Shared PyTorch training loop

In [14]:
def train_torch(model, Xtr, ytr, Xva, yva, Xte, yte, epochs=40, bs=64, lr=1e-3, wd=1e-4, patience=6):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.BCEWithLogitsLoss()
    tr = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr).float()), batch_size=bs, shuffle=True)
    best_va, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit = model(xb).squeeze(-1)
            loss = loss_fn(logit, yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p_va = torch.sigmoid(model(torch.from_numpy(Xva).to(DEVICE)).squeeze(-1)).cpu().numpy()
        va_auc = roc_auc_score(yva, p_va)
        if va_auc > best_va: best_va, best_state, bad = va_auc, {k:v.clone() for k,v in model.state_dict().items()}, 0
        else: bad += 1
        if bad >= patience: break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        p_tr = torch.sigmoid(model(torch.from_numpy(Xtr).to(DEVICE)).squeeze(-1)).cpu().numpy()
        p_te = torch.sigmoid(model(torch.from_numpy(Xte).to(DEVICE)).squeeze(-1)).cpu().numpy()
    return {'train_acc': accuracy_score(ytr, p_tr>0.5),
            'val_auc': best_va,
            'test_acc': accuracy_score(yte, p_te>0.5),
            'test_auc': roc_auc_score(yte, p_te),
            'test_f1':  f1_score(yte, p_te>0.5)}

## 10. Model 2 — BiGRU (FIXED: last hidden state, not avg pool)

In [15]:
class BiGRUClf(nn.Module):
    def __init__(self, n_in, h=48, layers=2, drop=0.3):
        super().__init__()
        self.gru = nn.GRU(n_in, h, num_layers=layers, batch_first=True, bidirectional=True,
                          dropout=drop if layers > 1 else 0)
        self.head = nn.Sequential(nn.LayerNorm(2*h), nn.Dropout(drop), nn.Linear(2*h, 1))
    def forward(self, x):
        out, _ = self.gru(x)        # (B, T, 2h)
        last = out[:, -1, :]        # use LAST step, not avg
        return self.head(last)

res_bigru = train_torch(BiGRUClf(Xtr_s.shape[-1]), Xtr_s, ytr_s, Xva_s, yva_s, Xte_s, yte_s)
print('BiGRU:', res_bigru)

BiGRU: {'train_acc': 0.6489451476793249, 'val_auc': np.float64(0.6653838152693987), 'test_acc': 0.6013986013986014, 'test_auc': np.float64(0.6397296538446111), 'test_f1': 0.48516129032258065}


## 11. Model 3 — BiRNN + Attention

In [16]:
class BiRNNAttn(nn.Module):
    def __init__(self, n_in, h=48, layers=1, drop=0.3):
        super().__init__()
        self.rnn = nn.GRU(n_in, h, num_layers=layers, batch_first=True, bidirectional=True,
                          dropout=drop if layers > 1 else 0)
        self.attn = nn.Linear(2*h, 1)
        self.head = nn.Sequential(nn.LayerNorm(2*h), nn.Dropout(drop), nn.Linear(2*h, 1))
    def forward(self, x):
        out, _ = self.rnn(x)                         # (B, T, 2h)
        a = torch.softmax(self.attn(out), dim=1)     # (B, T, 1)
        ctx = (a * out).sum(dim=1)                   # (B, 2h)
        return self.head(ctx)

res_attn = train_torch(BiRNNAttn(Xtr_s.shape[-1]), Xtr_s, ytr_s, Xva_s, yva_s, Xte_s, yte_s)
print('BiRNN+Attn:', res_attn)

BiRNN+Attn: {'train_acc': 0.5776371308016878, 'val_auc': np.float64(0.5727464754924709), 'test_acc': 0.5464535464535465, 'test_auc': np.float64(0.5618868075889455), 'test_f1': 0.42385786802030456}


## 12. Model 4 — BiRNN + Skip connection

In [17]:
class BiRNNSkip(nn.Module):
    def __init__(self, n_in, h=48, drop=0.3):
        super().__init__()
        self.rnn1 = nn.GRU(n_in, h, batch_first=True, bidirectional=True)
        self.rnn2 = nn.GRU(2*h, h, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(n_in, 2*h)  # to match skip dim
        self.drop = nn.Dropout(drop)
        self.head = nn.Sequential(nn.LayerNorm(2*h), nn.Linear(2*h, 1))
    def forward(self, x):
        r1, _ = self.rnn1(x)                  # (B, T, 2h)
        skip = self.proj(x)                   # raw-input skip
        r1 = self.drop(r1 + skip)
        r2, _ = self.rnn2(r1)                 # (B, T, 2h)
        r2 = r2 + r1                          # stage-2 skip
        return self.head(r2[:, -1, :])

res_skip = train_torch(BiRNNSkip(Xtr_s.shape[-1]), Xtr_s, ytr_s, Xva_s, yva_s, Xte_s, yte_s)
print('BiRNN+Skip:', res_skip)

BiRNN+Skip: {'train_acc': 0.679324894514768, 'val_auc': np.float64(0.7056256100878526), 'test_acc': 0.6233766233766234, 'test_auc': np.float64(0.6505354779190565), 'test_f1': 0.5527876631079478}


## 13. Summary

In [18]:
rows = [
    ('BEFORE — LogReg / 1-day target', before['acc'], before['auc'], before['f1'], None),
    ('AFTER  — LogReg / 5-day target', after_target['acc'], after_target['auc'], after_target['f1'], None),
    ('AFTER  — ARX (full features, 5d)', res_arx['test_acc'], res_arx['test_auc'], res_arx['test_f1'], res_arx['train_acc']),
    ('AFTER  — BiGRU (fixed pooling, 5d)', res_bigru['test_acc'], res_bigru['test_auc'], res_bigru['test_f1'], res_bigru['train_acc']),
    ('AFTER  — BiRNN+Attn (5d)', res_attn['test_acc'], res_attn['test_auc'], res_attn['test_f1'], res_attn['train_acc']),
    ('AFTER  — BiRNN+Skip (5d)', res_skip['test_acc'], res_skip['test_auc'], res_skip['test_f1'], res_skip['train_acc']),
]
summary = pd.DataFrame(rows, columns=['model','test_acc','test_auc','test_f1','train_acc'])
print(summary.to_string(index=False))
summary.to_csv('/content/drive/MyDrive/Quants /phase0_summary.csv', index=False)

                             model  test_acc  test_auc  test_f1  train_acc
    BEFORE — LogReg / 1-day target  0.707150  0.778479 0.673937        NaN
    AFTER  — LogReg / 5-day target  0.743389  0.801807 0.706935        NaN
  AFTER  — ARX (full features, 5d)  0.743389  0.801807 0.706935   0.741807
AFTER  — BiGRU (fixed pooling, 5d)  0.601399  0.639730 0.485161   0.648945
          AFTER  — BiRNN+Attn (5d)  0.546454  0.561887 0.423858   0.577637
          AFTER  — BiRNN+Skip (5d)  0.623377  0.650535 0.552788   0.679325
